In [4]:
!pip install "labtasker[plugins]" pandas

Looking in indexes: https://artifactory-cloud.chehejia.com/artifactory/api/pypi/liauto-pypi-l5/simple


In [1]:
import os
os.chdir("/path/to/starVLA")
print(os.getcwd())
from labtasker import ls_tasks

/path/to/starVLA


In [2]:
!labtasker task count -f 'metadata.benchmark == "LIBERO-plus"'

🟡 Pending: 43
🔵 Running: 4
🟢 Success: 13
🔴 Failed: 0
⚪ Cancelled: 0


In [3]:
# Filter by benchmark to avoid mixing with LIBERO tasks
# Add more conditions as needed:
# e.g. extra_filter = 'metadata.benchmark == "LIBERO-plus" and metadata.ckpt_stem == "steps_30000_pytorch_model"'
extra_filter = 'metadata.benchmark == "LIBERO-plus"'

tasks = ls_tasks(
    status="success",
    extra_filter=extra_filter if extra_filter else None,
    limit=1000,
).content
print(f"{len(tasks)} successful tasks")

13 successful tasks


In [4]:
tasks[0]

Task(unknown_fields={}, task_id='3f3362f7-c2f2-4b9d-9c65-30c5daead983', queue_id='b5b7ed91-b5ef-46bc-a244-33216576b5d9', status='success', task_name='steps_30000_pytorch_model_libero_object_0_210', created_at=datetime.datetime(2026, 4, 29, 10, 9, 56, 762000), start_time=datetime.datetime(2026, 4, 29, 11, 19, 27, 353000), last_heartbeat=datetime.datetime(2026, 4, 30, 6, 26, 32, 813000), last_modified=datetime.datetime(2026, 4, 30, 6, 26, 33, 818000), heartbeat_timeout=30.0, task_timeout=None, max_retries=3, retries=0, priority=10, metadata={'benchmark': 'LIBERO-plus', 'ckpt_stem': 'steps_30000_pytorch_model', 'task_suite': 'libero_object'}, args={'ckpt': '/path/to/starVLA/playground/Pretrained_models/StarVLA/Qwen2.5-VL-OFT-LIBERO-4in1/checkpoints/steps_30000_pytorch_model.pt', 'task_suite': 'libero_object', 'start_idx': 0, 'end_idx': 210}, cmd=['examples/LIBERO-plus/eval_files/parallel_eval_labtasker/run.py', '--env', 'examples/LIBERO-plus/eval_files/parallel_eval_labtasker/.env'], summ

In [5]:
import pandas as pd

# Each task is one slice [start_idx, end_idx); aggregate slices → per-(ckpt, suite) totals
rows = []
for t in tasks:
    rows.append({
        "ckpt_stem":       t.metadata.get("ckpt_stem", ""),
        "task_suite":      t.summary["task_suite"],
        "start_idx":       t.args.get("start_idx"),
        "end_idx":         t.args.get("end_idx"),
        "total_episodes":  t.summary["total_episodes"],
        "total_successes": t.summary["total_successes"],
    })

raw_df = pd.DataFrame(rows)

# Sum slices → per-(ckpt_stem, task_suite)
suite_df = (
    raw_df
    .groupby(["ckpt_stem", "task_suite"])[["total_episodes", "total_successes"]]
    .sum()
)
suite_df["success_rate"] = suite_df["total_successes"] / suite_df["total_episodes"]
suite_df["n_slices"] = raw_df.groupby(["ckpt_stem", "task_suite"]).size()
suite_df = suite_df.sort_index()
suite_df

total_episodes  total_successes  \
ckpt_stem                 task_suite                                        
steps_30000_pytorch_model libero_object            73450            48456   
                          libero_spatial           60100            40580   

                                          success_rate  n_slices  
ckpt_stem                 task_suite                              
steps_30000_pytorch_model libero_object       0.659714         7  
                          libero_spatial      0.675208         6

In [6]:
# Total tasks per suite in LIBERO-plus (from benchmark; update if suite changes)
SUITE_SIZES = {
    "libero_10":      2519,
    "libero_goal":    2591,
    "libero_object":  2518,
    "libero_spatial": 2402,
}

# Join with suite_df to show "X successes / Y episodes across Z tasks"
size_series = pd.Series(SUITE_SIZES, name="total_tasks").rename_axis("task_suite")

# suite_df has multi-index (ckpt_stem, task_suite); join on task_suite level
suite_full = suite_df.join(size_series, on="task_suite")
suite_full["tasks_per_episode"] = suite_full["total_episodes"] / suite_full["total_tasks"]  # ≈ num_trials_per_task
suite_full = suite_full[["total_tasks", "total_episodes", "total_successes", "success_rate", "n_slices", "tasks_per_episode"]]
suite_full

total_tasks  total_episodes  \
ckpt_stem                 task_suite                                    
steps_30000_pytorch_model libero_object          2518           73450   
                          libero_spatial         2402           60100   

                                          total_successes  success_rate  \
ckpt_stem                 task_suite                                      
steps_30000_pytorch_model libero_object             48456      0.659714   
                          libero_spatial            40580      0.675208   

                                          n_slices  tasks_per_episode  
ckpt_stem                 task_suite                                   
steps_30000_pytorch_model libero_object          7          29.169976  
                          libero_spatial         6          25.020816

In [7]:
# Per-checkpoint aggregate across all suites (slices already merged in suite_df)
summary_rows = []
for ckpt_stem, grp in suite_df.groupby("ckpt_stem"):
    total_ep   = grp["total_episodes"].sum()
    total_succ = grp["total_successes"].sum()
    summary_rows.append({
        "ckpt_stem":       ckpt_stem,
        "n_suites":        len(grp),
        "total_episodes":  int(total_ep),
        "total_successes": int(total_succ),
        "success_rate":    total_succ / total_ep if total_ep > 0 else 0.0,
    })

summary_df = pd.DataFrame(summary_rows).set_index("ckpt_stem")
summary_df.sort_values("success_rate", ascending=False)

,n_suites,total_episodes,total_successes,success_rate
ckpt_stem,,,,
steps_30000_pytorch_model,2,133550,89036,0.666687


In [8]:
# Per-category breakdown (LIBERO-plus disturbance categories)
# category_results: {category: {total_count, success_count}}
from collections import defaultdict

cat_agg: dict[str, dict] = defaultdict(lambda: {"total_count": 0, "success_count": 0})
for t in tasks:
    for cat, counts in t.summary.get("category_results", {}).items():
        cat_agg[cat]["total_count"]   += counts["total_count"]
        cat_agg[cat]["success_count"] += counts["success_count"]

cat_rows = [
    {
        "category":     cat,
        "total":        v["total_count"],
        "successes":    v["success_count"],
        "success_rate": v["success_count"] / v["total_count"] if v["total_count"] > 0 else 0.0,
    }
    for cat, v in cat_agg.items()
]
cat_df = pd.DataFrame(cat_rows).set_index("category").sort_values("success_rate", ascending=True)
cat_df

,total,successes,success_rate
category,,,
Objects Layout,0,0,0.000000
Sensor Noise,0,0,0.000000
Camera Viewpoints,38600,14019,0.363187
Robot Initial States,37400,22149,0.592219
Language Instructions,21800,18926,0.868165
Light Conditions,10450,9877,0.945167
Background Textures,25300,24065,0.951186
